# Spicy fish: discovery and operational resilience

This case has two distinct evidence levels: the founder’s account of business decisions and closure, and synthetic public calculations. The sample does not prove the real outcomes. The scenario below is hypothetical and does not describe the actual weather event.


## 1. Validate what the sample can support

Run from any notebook working directory. The schema checks live in the shared script that also generates the repository charts. The label `new` is a constructed familiarity group, not a verified unique or first-time buyer.


In [1]:
import sys
from pathlib import Path
root = Path.cwd()
if not (root / "scripts" / "analyze.py").exists():
    root = root.parent
if not (root / "scripts" / "analyze.py").exists():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory")
sys.path.insert(0, str(root / "scripts"))
from analyze import load_and_validate
df = load_and_validate()
print(f"Validated {len(df):,} synthetic orders; {df.units.sum():,} synthetic units")


Validated 2,330 synthetic orders; 2,993 synthetic units


## 2. Test the narrow-audience assumption within the illustration

Denominator: **orders**, never unique people. This sample has no customer ID or method for verifying familiarity.


In [2]:
mix = df.customer_familiarity.value_counts().reindex(["new", "familiar"])
print((mix / len(df) * 100).round(1).to_string())


customer_familiarity
new         60.6
familiar    39.4


In the synthetic example 60.6% of **orders** have the `new` label. This is a demonstration of segmentation, not evidence that 60.6% of the real business’s buyers were new.


## 3. Compare the source labels within each segment

`repeat_purchase` is an order-behavior label mixed into the source field, and 92 rows have both `new` and `repeat_purchase`. This is a data definition problem: do not call these verified acquisition channels or count distinct first-time buyers.


In [3]:
counts = __import__("pandas").crosstab(df.customer_familiarity, df.acquisition_channel)
shares = (counts.div(counts.sum(axis=1), axis=0) * 100).round(1)
print(shares.to_string())
print("Rows labeled both new and repeat_purchase:", counts.loc["new", "repeat_purchase"])


acquisition_channel   referral  repeat_purchase  search  tiktok_creator
customer_familiarity                                                   
familiar                   9.7             29.3    38.6            22.4
new                        9.1              6.5    12.3            72.0
Rows labeled both new and repeat_purchase: 92


Within the constructed `new` group, 72.0% of synthetic orders carry the `tiktok_creator` label. This cannot establish creator exposure, incrementality, or the size of a real market.


## 4. Inspect the sample trend without inventing a turning point

The supplied monthly series rises throughout. Its original generator is unavailable, and the series contains neither real marketing intervention dates nor weather events. No before/after marker or causal conclusion is justified.


In [4]:
trend = df.groupby("month", sort=True).agg(orders=("order_id", "count"), units=("units", "sum"))
print(trend.to_string())


         orders  units
month                 
2025-11     180    227
2025-12     210    275
2026-01     260    341
2026-02     420    542
2026-03     560    704
2026-04     700    904


## 5. A hypothetical inventory response, not a reconstruction

We cannot estimate the actual closure from these orders. Instead, define a simple delivery-delay scenario that makes the operational question testable. Assumptions: 10 units of daily demand for 21 days, 120 initially on hand, and a shipment of 100 due on day 9. Shipment arrives before that day’s demand. If stock runs out, demand is unfilled rather than backordered.


In [5]:
from scenario import Assumptions, simulate
for label, case in {"On time": Assumptions(), "Seven-day delay": Assumptions(delay_days=7), "Delay + 30-unit buffer": Assumptions(delay_days=7, opening_stock=150)}.items():
    rows = simulate(case)
    print(f"{label}: fulfilled={sum(r[1] for r in rows)}, unfilled={sum(r[2] for r in rows)}")


On time: fulfilled=210, unfilled=0
Seven-day delay: fulfilled=180, unfilled=30
Delay + 30-unit buffer: fulfilled=210, unfilled=0


![Hypothetical delivery-delay stress test](../charts/disruption_scenario.png)

In this scenario a seven-day delay yields 30 unfilled units. Adding 30 opening units removes the shortfall within this chosen horizon. Extra stock ties up cash and could expire. Changing assumptions changes the result. No weather mechanism, actual losses, or prevention claim is inferred. A production analysis would need order dates, inventory snapshots, lead times, supplier and cost data, and dated disruption records.


## 6. Vary the assumptions

A single illustrative scenario is fragile. Check unfilled units under several delay and inventory assumptions; zero means this simple model can fulfill all demand over its 21-day horizon.


In [6]:
for delay in (0, 3, 7, 10):
    losses = [sum(row[2] for row in simulate(Assumptions(delay_days=delay, opening_stock=stock))) for stock in (120, 135, 150)]
    print(f"delay={delay:2} days | opening stock 120 / 135 / 150: {losses}")


delay= 0 days | opening stock 120 / 135 / 150: [0, 0, 0]
delay= 3 days | opening stock 120 / 135 / 150: [0, 0, 0]
delay= 7 days | opening stock 120 / 135 / 150: [30, 15, 0]
delay=10 days | opening stock 120 / 135 / 150: [60, 45, 30]


This demonstrates a tradeoff, not a recommendation: additional stock ties up cash and may expire. The real business lacked the publicly available dated operations and disruption records needed to choose or evaluate a policy. The closure changes the next question I would ask; it does not make this synthetic calculation a retrospective explanation.
